# Correction - Perceptron binaire et descente de gradient

On reprend le même dataset et on complète les fonctions manquantes pour entraîner un petit perceptron.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

students = pd.DataFrame({
    'hours': [1.5, 2.0, 3.0, 4.5, 5.0, 6.0, 7.5, 8.0],
    'attendance': [30, 35, 40, 55, 60, 70, 80, 90],
    'validated': [0, 0, 0, 0, 1, 1, 1, 1],
})

X = students[['hours', 'attendance']]
y = students['validated']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

## 1. Visualisation

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(
    students['hours'],
    students['attendance'],
    c=students['validated'],
    cmap='coolwarm',
    s=90,
    edgecolor='black'
)
plt.xlabel('Hours')
plt.ylabel('Attendance')
plt.title('Dataset étudiants')
plt.show()

## 2. Standardisation

In [ ]:
mu = X_train.mean()
sigma = X_train.std(ddof=0)

X_train_scaled = (X_train - mu) / sigma
X_test_scaled = (X_test - mu) / sigma

X_train_scaled

## 3. Sigmoïde et prédiction

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def predict_proba(X, w, b):
    z = np.dot(X, w) + b
    return sigmoid(z)

def predict_class(X, w, b):
    proba = predict_proba(X, w, b)
    return (proba >= 0.5).astype(int)

print(sigmoid(0))
print(sigmoid(2))

## 4. Fonction de coût

In [ ]:
def binary_cross_entropy(y_true, y_pred):
    eps = 1e-12
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# vérification rapide
print(binary_cross_entropy(np.array([0, 1]), np.array([0.1, 0.9])))

## 5. Descente de gradient

In [ ]:
def train_perceptron(X, y, epochs=2000, eta=0.1):
    X = X.to_numpy(dtype=float)
    y = y.to_numpy(dtype=float)
    w = np.zeros(X.shape[1])
    b = 0.0
    losses = []

    for _ in range(epochs):
        z = np.dot(X, w) + b
        y_pred = sigmoid(z)

        loss = binary_cross_entropy(y, y_pred)
        losses.append(loss)

        error = y_pred - y
        grad_w = np.dot(X.T, error) / len(X)
        grad_b = np.mean(error)

        w = w - eta * grad_w
        b = b - eta * grad_b

    return w, b, losses

w, b, losses = train_perceptron(X_train_scaled, y_train)
print(w)
print(b)

## 6. Courbe de perte

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Descente de gradient')
plt.grid(alpha=0.2)
plt.show()

## 7. Évaluation

In [ ]:
y_pred = predict_class(X_test_scaled.to_numpy(dtype=float), w, b)
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print('accuracy =', acc)
print(cm)

## 8. Résultat à retenir

Le modèle apprend :

- des poids pour chaque variable ;
- un biais pour déplacer la séparation ;
- une probabilité via la sigmoïde ;
- une mise à jour itérative grâce à la descente de gradient.

C'est la mécanique de base d'un neurone artificiel.